In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from the_well.data import WellDataset
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

from modules import *
from datasets import TRMDataset
from vis_tools import GIF

In [1]:
SEED       = 42
EPOCHS     = 15
BATCH_SIZE = 16

MODES1     = 16   # Modos de Fourier na primeira dimensão espacial (x)
MODES2     = 16   # Modos de Fourier na segunda dimensão espacial (y)
WIDTH      = 32   # Número de canais internos (largura do modelo)
DEPTH      = 4    # Quantidade de camadas de Fourier
PROJ_DIM   = 128  # Dimensão da MLP de projeção para a saída

In [3]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")
print(f"Torch: {torch.__version__}  |  Torchvision: {__import__('torchvision').__version__}")

Dispositivo: cuda
Torch: 2.11.0+cu128  |  Torchvision: 0.26.0+cu128


In [ ]:
train_dataset = WellDataset(
    path="/mnt/storage_C1/BILL_pino/the_well/datasets/turbulent_radiative_layer_2D",
    well_split_name="train"
)

validation_dataset = WellDataset(
    path="/mnt/storage_C1/BILL_pino/the_well/datasets/turbulent_radiative_layer_2D",
    well_split_name="valid"
)

train_ds = TRMDataset(train_dataset)
val_ds = TRMDataset(validation_dataset)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=16,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)

In [8]:
x, y = train_ds[0]
in_dim, out_dim = x.shape[-1], y.shape[-1] 

model = FNO2d(
    modes1=MODES1,
    modes2=MODES2,
    width=WIDTH,
    in_dim=in_dim,
    out_dim=out_dim,
    depth=DEPTH,
    proj_dim=PROJ_DIM
).cuda()
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

criterion = relative_l2_loss

In [9]:
x, truth = train_ds[0]

print(truth.shape)

with torch.no_grad():
    pred = model(x.unsqueeze(0).to(device))

print(pred.shape)

torch.Size([128, 384, 4])
torch.Size([1, 128, 384, 4])


In [10]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS
)

Epoch 1/15: 100%|██████████| 5/5 [00:08<00:00,  1.71s/it, loss=0.995394]


Epoch 001 | train = 0.995394 | val = 0.986478 | best_val = 0.986478


Epoch 2/15: 100%|██████████| 5/5 [00:01<00:00,  3.24it/s, loss=0.973108]


Epoch 002 | train = 0.973108 | val = 0.941962 | best_val = 0.941962


Epoch 3/15: 100%|██████████| 5/5 [00:01<00:00,  3.11it/s, loss=0.898992]


...

Epoch 013 | train = 0.077361 | val = 0.076129 | best_val = 0.076129


Epoch 14/15: 100%|██████████| 5/5 [00:01<00:00,  3.32it/s, loss=0.076042]


Epoch 014 | train = 0.076042 | val = 0.075775 | best_val = 0.075775


Epoch 15/15: 100%|██████████| 5/5 [00:01<00:00,  3.34it/s, loss=0.075825]


Epoch 015 | train = 0.075825 | val = 0.075549 | best_val = 0.075549


In [ ]:
torch.save(model.state_dict(), "/mnt/storage_C1/BILL_pino/TRM15-8-16_16-32-4-128_test.pth")

In [ ]:
train_dataset[0]["input_fields"].shape
len(train_dataset)

72

In [ ]:
times = []

for i in range(500):
    times.append(train_ds.ds[i]["output_time_grid"].item())

print(sorted(set(times)))

RuntimeError: a Tensor with 100 elements cannot be converted to Scalar

In [ ]:
import inspect


print(inspect.signature(WellDataset))

(path: Optional[str] = None, normalization_path: Optional[str] = None, well_base_path: Optional[str] = None, well_dataset_name: Optional[str] = None, well_split_name: Literal['train', 'valid', 'test', None] = None, include_filters: List[str] = [], exclude_filters: List[str] = [], use_normalization: bool = False, normalization_type: Optional[Callable[..., Any]] = None, max_rollout_steps=100, n_steps_input: int = 1, n_steps_output: int = 1, min_dt_stride: int = 1, max_dt_stride: int = 1, flatten_tensors: bool = True, restrict_num_trajectories: Union[float, int, NoneType] = None, restrict_num_samples: Union[float, int, NoneType] = None, restriction_seed: int = 0, cache_small: bool = True, max_cache_size: float = 1000000000.0, return_grid: bool = True, normalize_time_grid: bool = True, boundary_return_type: str = 'padding', full_trajectory_mode: bool = False, start_output_steps_at_t: int = -1, name_override: Optional[str] = None, transform: Optional[ForwardRef('Augmentation')] = None, min_